# 🚀 SupportHR CV Industry Classifier - Google Colab Deployment

Notebook này dùng để triển khai Model Pipeline phân loại ngành nghề CV (24 categories) lên **Google Colab**, tận dụng GPU/CPU miễn phí và mở đường hầm HTTPS công khai qua **Cloudflare Tunnel** để đấu nối trực tiếp vào API chung của SupportHR (`cv-match-api`).

### Bước 1: Cài đặt các thư viện cần thiết

In [ ]:
!pip install -q fastapi uvicorn scikit-learn joblib pycloudflared pydantic requests pandas

### Bước 2: Tạo mã nguồn FastAPI Microservice trên Colab

In [ ]:
%%writefile server.py
import os
import re
import math
from pathlib import Path
from typing import Any
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
import joblib

DEFAULT_LABELS = [
    "ACCOUNTANT", "ADVOCATE", "AGRICULTURE", "APPAREL", "ARTS", "AUTOMOBILE",
    "AVIATION", "BANKING", "BPO", "BUSINESS-DEVELOPMENT", "CHEF", "CONSTRUCTION",
    "CONSULTANT", "DESIGNER", "DIGITAL-MEDIA", "ENGINEERING", "FINANCE", "FITNESS",
    "HEALTHCARE", "HR", "INFORMATION-TECHNOLOGY", "PUBLIC-RELATIONS", "SALES", "TEACHER"
]

app = FastAPI(title="SupportHR Colab Classifier")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

_model = None
_model_source = os.getenv("MODEL_SOURCE", "colab://t4-gpu-classifier-v1")

class ClassifyRequest(BaseModel):
    cv_text: str = Field(..., description="Raw text extracted from CV")
    top_k: int = Field(default=3, ge=1, le=24)

def clean_text(text: str) -> str:
    normalized = str(text).lower()
    normalized = re.sub(r"[^\w\s]+", " ", normalized, flags=re.UNICODE)
    normalized = normalized.replace("_", " ")
    return re.sub(r"\s+", " ", normalized).strip()

def _softmax(values: list[float]) -> list[float]:
    if not values:
        return []
    max_val = max(values)
    exps = [math.exp(v - max_val) for v in values]
    total = sum(exps)
    return [v / total for v in exps] if total > 0 else [0.0 for _ in values]

def init_model():
    global _model
    model_file = Path("text_classifier_model.pkl")
    if model_file.is_file():
        _model = joblib.load(model_file)
        print("Loaded existing text_classifier_model.pkl")
        return
    print("Training in-memory TF-IDF + LinearSVC pipeline for SupportHR 24 categories...")
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.pipeline import Pipeline
    from sklearn.svm import LinearSVC
    seed_data = [
        ("accounting finance ledger tax audit balance sheet accountant bookkeeping reconciliation", "ACCOUNTANT"),
        ("legal advocate court lawyer litigation litigation counsel compliance contract law", "ADVOCATE"),
        ("agriculture crop farming agronomy harvesting soil seeds livestock irrigation", "AGRICULTURE"),
        ("apparel fashion textile garment clothing designer merchandising pattern fabrication", "APPAREL"),
        ("fine arts painting sculpture gallery illustrator artist visual creative exhibition", "ARTS"),
        ("automobile vehicle automotive mechanic engine car repair transport diagnostic maintenance", "AUTOMOBILE"),
        ("aviation pilot flight aircraft airline aeronautical aerospace crew navigation avionics", "AVIATION"),
        ("banking loan credit deposit investment branch banking banker mortgage retail banking", "BANKING"),
        ("bpo customer support call center outsourcing telemarketing agent technical support inbound", "BPO"),
        ("business development partnership growth strategy b2b sales lead enterprise account", "BUSINESS-DEVELOPMENT"),
        ("chef culinary cooking cuisine kitchen restaurant pastry food recipe hospitality menu", "CHEF"),
        ("civil engineering construction architect building site contractor structural inspection", "CONSTRUCTION"),
        ("consulting advisory management consultant strategy operational transformation roadmap", "CONSULTANT"),
        ("graphic designer ui ux figma product layout typography visual wireframing prototyping", "DESIGNER"),
        ("digital media social content video creator broadcast marketing journalism podcast", "DIGITAL-MEDIA"),
        ("electrical mechanical engineering firmware hardware maintenance cad plc electronics", "ENGINEERING"),
        ("financial analyst equity wealth investment banking portfolio risk valuation treasury", "FINANCE"),
        ("fitness trainer gym coach exercise workout health sports athletics personal training", "FITNESS"),
        ("doctor nurse medical hospital clinical patient healthcare therapy pharmacy physician", "HEALTHCARE"),
        ("human resources talent acquisition recruitment payroll hr screening onboarding employee relations", "HR"),
        ("python fastapi react typescript software engineer developer backend frontend cloud devops fullstack", "INFORMATION-TECHNOLOGY"),
        ("public relations communications press release media spokesperson crisis branding communications", "PUBLIC-RELATIONS"),
        ("sales executive quota pipeline negotiation account manager cold calling b2b revenue", "SALES"),
        ("teacher tutor education curriculum classroom school professor pedagogy lecturing academic", "TEACHER"),
    ]
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
        ("clf", LinearSVC(random_state=42, dual="auto")),
    ])
    pipeline.fit([d[0] for d in seed_data], [d[1] for d in seed_data])
    _model = pipeline
    joblib.dump(_model, "text_classifier_model.pkl")
    print("Model initialized & saved.")

@app.on_event("startup")
def startup():
    init_model()

@app.get("/health")
def health():
    return {"status": "ok", "ready": _model is not None}

@app.get("/api/cv/classifier-status")
@app.get("/api/classifier-status")
def classifier_status():
    classes = [str(c) for c in getattr(_model, "classes_", DEFAULT_LABELS)] if _model else []
    return {
        "ready": _model is not None,
        "model_source": _model_source,
        "label_count": len(classes),
        "labels": classes,
        "error": None if _model else "Model not ready"
    }

@app.post("/api/cv/classify-industry")
@app.post("/api/classify-industry")
def classify_industry(payload: ClassifyRequest):
    if _model is None:
        raise HTTPException(status_code=503, detail="Model not ready")
    cleaned = clean_text(payload.cv_text)
    if not cleaned:
        raise HTTPException(status_code=400, detail="CV text empty")
    classes = [str(c) for c in getattr(_model, "classes_", DEFAULT_LABELS)]
    if hasattr(_model, "decision_function"):
        scores = _model.decision_function([cleaned])
        if hasattr(scores, "tolist"):
            scores = scores.tolist()
        if isinstance(scores, list) and scores and isinstance(scores[0], list):
            scores = scores[0]
        probs = _softmax([float(v) for v in scores])
    elif hasattr(_model, "predict_proba"):
        probs = _model.predict_proba([cleaned])[0]
    else:
        probs = [1.0 if c == _model.predict([cleaned])[0] else 0.0 for c in classes]
    scored = sorted([{"label": l, "score": round(float(s), 4)} for l, s in zip(classes, probs)], key=lambda x: x["score"], reverse=True)[:payload.top_k]
    return {
        "predicted_label": scored[0]["label"] if scored else "UNKNOWN",
        "confidence": scored[0]["score"] if scored else 0.0,
        "top_predictions": scored,
        "model_source": _model_source,
    }

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)


### Bước 3: Khởi động Server và Mở Đường Hầm Cloudflare Tunnel

In [ ]:
import subprocess
import time
from pycloudflared import try_cloudflare
import requests

# Khởi động FastAPI server ngầm trên port 8000
server_process = subprocess.Popen(["python", "server.py"])
print("⏳ Đang khởi động FastAPI server...")
time.sleep(3)

# Mở Cloudflare Tunnel (Miễn phí, có HTTPS, không cần tài khoản)
tunnel = try_cloudflare(port=8000)
public_url = tunnel.tunnel.rstrip("/")

print("\n" + "="*60)
print("✅ SERVER ĐÃ SẴN SÀNG!")
print(f"🌐 Public HTTPS URL: {public_url}")
print("="*60)

# Tự kiểm tra trạng thái server
try:
    res = requests.get(f"{public_url}/health", timeout=10)
    print(f"Health Check qua Tunnel: {res.json()}")
except Exception as e:
    print(f"Cảnh báo tunnel test: {e}")

print("\n📝 Cấu hình dán vào file api_server/.env của SupportHR:")
print(f"LOCAL_CLASSIFIER_MODE=auto")
print(f"LOCAL_CLASSIFIER_REMOTE_CLASSIFY_URL={public_url}/api/cv/classify-industry")
print(f"LOCAL_CLASSIFIER_REMOTE_STATUS_URL={public_url}/api/cv/classifier-status")
print(f"LOCAL_CLASSIFIER_REMOTE_TIMEOUT_SECONDS=10.0")


### Bước 4: Test phân loại thử nghiệm trực tiếp trên Colab

In [ ]:
test_cv = """
Senior Fullstack Software Engineer with 7 years of hands-on experience building
distributed web applications with Python FastAPI, TypeScript, React, Docker, and PostgreSQL.
Specialized in backend scalability, Redis queues, and CI/CD pipelines.
"""

response = requests.post(
    f"{public_url}/api/cv/classify-industry",
    json={"cv_text": test_cv, "top_k": 3}
)
print("Kết quả dự đoán:")
print(response.json())
